# NTX Q1 Stage 18 — AgentDojo Guarded External Evaluation

Repairs the prior AgentDojo infrastructure gap. It installs the official package, inventories suites, ingests official native case JSON, and computes guarded-vs-unguarded paired metrics without converting infrastructure failures into model scores.

**Evidence rule:** failed/incomplete third-party infrastructure is recorded as such; no missing score is converted into a model failure or a success.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time, hashlib, zipfile, random
BUNDLE = Path(os.environ.get("NTX_NOTEBOOK_BUNDLE", ".")).resolve()
if str(BUNDLE) not in sys.path:
    sys.path.insert(0, str(BUNDLE))
from ntx_q1_shared import *
ROOT = ensure_dir(DEFAULT_ROOT)
RESULTS_ROOT = ensure_dir(DEFAULT_RESULTS)
SEED = int(os.environ.get("NTX_SEED", "42"))
random.seed(SEED)
print("ROOT:", ROOT)
print("RESULTS:", RESULTS_ROOT)
print("Frozen repo commit:", REPO_COMMIT)

In [ ]:
import pandas as pd, numpy as np
STAGE="18_agentdojo_guarded"
OUT=ensure_dir(RESULTS_ROOT/STAGE)
INP=ensure_dir(ROOT/"inputs"/STAGE)
SELFTEST=os.environ.get("NTX_SELFTEST","0")=="1"

if not SELFTEST:
    cp=run([sys.executable,"-m","pip","install","-q","agentdojo"],check=False,timeout=300)
    (OUT/"agentdojo_install.log").write_text(cp.stdout,encoding="utf-8")
    install_ok=(cp.returncode==0)
else:
    install_ok=False

write_json(OUT/"agentdojo_install_status.json",{"install_ok":install_ok,"selftest":SELFTEST})

In [ ]:
# Dynamically inventory official suites without assuming a specific CLI.
inventory=[]
if install_ok:
    try:
        from agentdojo.task_suite.load_suites import get_suite
        for suite_name in ["banking","workspace","travel","slack"]:
            try:
                s=get_suite("v1.2.2",suite_name)
                inventory.append({
                    "suite":suite_name,
                    "user_tasks":len(getattr(s,"user_tasks",{})),
                    "injection_tasks":len(getattr(s,"injection_tasks",{})),
                    "status":"OK"
                })
            except Exception as e:
                inventory.append({"suite":suite_name,"status":"FAIL","error":repr(e)})
    except Exception as e:
        inventory=[{"suite":"ALL","status":"FAIL","error":repr(e)}]
pd.DataFrame(inventory).to_csv(OUT/"agentdojo_suite_inventory.csv",index=False)
display(pd.DataFrame(inventory))

## Live-run handoff

This notebook accepts outputs from the official AgentDojo runner rather than hard-coding an unstable CLI.

Set `NTX_AGENTDOJO_EXPORT_ROOT` to the output directory produced by the official harness. Existing repository Stage 11/standalone notebooks may also be used to generate these files.

The parser below walks all JSON files and preserves official `utility` and `security` fields when present.

In [ ]:
export_root=os.environ.get("NTX_AGENTDOJO_EXPORT_ROOT","")
records=[]
failures=[]
if export_root and Path(export_root).exists():
    for p in Path(export_root).rglob("*.json"):
        try:
            obj=json.loads(p.read_text(encoding="utf-8"))
            objs=obj if isinstance(obj,list) else [obj]
            for x in objs:
                if not isinstance(x,dict): continue
                records.append({
                    "source_file":str(p),
                    "suite":x.get("suite") or x.get("task_suite") or p.parts[-5] if len(p.parts)>=5 else "",
                    "task_id":x.get("task_id") or x.get("id") or p.stem,
                    "utility":x.get("utility"),
                    "security":x.get("security"),
                    "model":x.get("model") or x.get("model_name"),
                    "condition":x.get("condition") or ("attack" if "injection" in str(p).lower() else "clean"),
                    "raw_json":json.dumps(x,ensure_ascii=False),
                })
        except Exception as e:
            failures.append({"file":str(p),"error":repr(e)})
else:
    print("Set NTX_AGENTDOJO_EXPORT_ROOT after running the official AgentDojo harness.")

native=pd.DataFrame(records)
native.to_csv(OUT/"agentdojo_native_cases.csv",index=False)
pd.DataFrame(failures).to_csv(OUT/"agentdojo_parse_failures.csv",index=False)
print("Parsed native cases:",len(native),"failures:",len(failures))

## Guard overlay input

For a defensible NiyamTrace authorization comparison, provide a paired CSV at `NTX_AGENTDOJO_GUARD_RESULTS` with:

`task_id, condition, unguarded_verdict, guarded_verdict, expected_verdict, guard_latency_ms`

The guarded verdict must come from the actual NiyamTrace runtime/adapter used in the experiment, not from this notebook.

In [ ]:
guard_file=os.environ.get("NTX_AGENTDOJO_GUARD_RESULTS","")
if guard_file and Path(guard_file).exists():
    g=pd.read_csv(guard_file)
    req={"task_id","condition","unguarded_verdict","guarded_verdict","expected_verdict"}
    assert req.issubset(g.columns), f"Missing {req-set(g.columns)}"
    mg=classification_metrics(g.rename(columns={"guarded_verdict":"actual_verdict"}))
    mu=classification_metrics(g.rename(columns={"unguarded_verdict":"actual_verdict"}))
    summary={"guarded":mg,"unguarded":mu,"n_pairs":int(len(g))}
    if "guard_latency_ms" in g:
        summary["guard_latency_median_ms"]=float(g.guard_latency_ms.median())
        summary["guard_latency_p95_ms"]=float(g.guard_latency_ms.quantile(.95))
    write_json(OUT/"agentdojo_guarded_summary.json",summary)
    g.to_csv(OUT/"agentdojo_guarded_pairs.csv",index=False)
    paper_ready=(len(g)>=50 and g.expected_verdict.notna().all() and set(["clean","attack"]).intersection(set(g.condition.astype(str).str.lower())))
    record_status(OUT,STAGE,"PASS" if paper_ready else "PARTIAL",paper_ready=bool(paper_ready),summary=summary)
    display(pd.DataFrame([{"condition":"unguarded",**mu},{"condition":"guarded",**mg}]))
else:
    status="INFRA_OR_GUARD_DATA_PENDING"
    record_status(OUT,STAGE,status,n_native=int(len(native)),install_ok=install_ok)

In [ ]:
z=package_stage(OUT,"NTX_Q1_18_AGENTDOJO_GUARDED_RESULTS.zip")
print("Created:",z)
maybe_colab_download(z)